# 04_export_and_validate

현재 기본 설정은 `CAMPAIGN="finetune_wd_zero_control"`, `EXECUTE=False`입니다. **03의 SurvFace MagFace (`training.weight_decay=0.0`) 40 epoch 학습 완료 후 `EXECUTE=True`로 실행하세요.** 03이 저장한 `finetune_wd_zero_v1/selection.json`을 읽어 40 epoch 완료 모델을 검증·export하고 `MODEL_INVENTORY`(`purpose="diagnostic_control"`)를 생성합니다.

03과 동일한 `CAMPAIGN` 및 `SELECTION_FILE`을 사용합니다.
- `CAMPAIGN="finetune_wd_zero_control"`: 완료된 `WD=0` Parent를 재사용하고 SurvFace 5,319명 MagFace의 fine-tune `training.weight_decay=0.0`만 대조한 40 epoch 완료 모델을 검증·export하고 `MODEL_INVENTORY`(`purpose="diagnostic_control"`)를 생성합니다.
- `CAMPAIGN="parent_control"`: `WD=0` Parent + 기존 fine-tune `weight_decay=0.0001` 대조 경로(`followup/selection.json`)를 검증·export합니다.
- `CAMPAIGN="ucface_review"` (`EXECUTE=False`): 현재 완료 체크포인트의 export 계획 및 UCFace 재현 조건을 점검합니다.
- `CAMPAIGN="paper_baseline"`: SurvFace 5,319명 논문 기준선 학습 완료 모델과 SCFace/TinyFace 완료 정렬 모델 8개를 검증·export하고 `MODEL_INVENTORY`(`purpose="paper_baseline"`)를 생성합니다.
- `CAMPAIGN="selected"`: 기존 연구 조건(SurvFace 4,255명 + 보정 인물 분리) 12개 완료 모델을 검증·export하고 `MODEL_INVENTORY`(`purpose="local_protocol_baseline"`)를 생성합니다.


## 1. 설정
00에서 저장한 설정을 이어받으며 HEADS·SEEDS·FINETUNE_DATASETS와 부모·backbone·경로는 03의 완료 모델과 같아야 합니다. 04는 학습하지 않고 완료된 backbone을 export합니다. 새 캠페인은 앞 단계를 먼저 완료해야 합니다. CosFace metadata의 head=cosface, source_config.head.type=AM_Softmax입니다. SETTINGS_PATH는 00과 같아야 합니다.

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [1]:
# cell 1 : 사용자 설정 및 캠페인 선택
# 목적: 03의 완료 모델과 일치하는 조건을 읽고 export 대상과 실행 여부를 지정합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
from modules.selected_campaign import default_plan, paper_baseline_plan
CAMPAIGN = "finetune_wd_zero_control"  # finetune_wd_zero_control=WD=0 Parent 재사용 + SurvFace MagFace fine-tune WD=0 대조 / parent_control=새 Parent 진단(ft WD=0.0001) / paper_baseline=SurvFace 5,319명 논문 기준선 / selected=기존 4,255명 보정분리 연구조건 / ucface_review=재현 검토 / baseline=과거 resize 캠페인
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)

SEEDS = list(saved["project"]["seed_list"])  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = list(saved["campaign"]["heads"])  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
SURVFACE_PAPER_RECIPE = "full_fr_resize_only"  # 03에서 사용한 paper_baseline recipe와 동일하게 맞춥니다.
SURVFACE_HEADS = ["magface"]  # 03에서 사용한 paper_baseline head 목록과 동일하게 맞춥니다.

CONTROL_ROOT = MODEL_ROOT / "facex/diagnostics/parent_weight_decay_zero_v1"
if CAMPAIGN == "finetune_wd_zero_control":
    SELECTED_ROOT = CONTROL_ROOT / "finetune_wd_zero_v1"
    SELECTION_FILE = SELECTED_ROOT / "selection.json"
elif CAMPAIGN == "parent_control":
    # 02a에서 40 epoch 완료 Parent를 검증해 발행한 선택 파일만 사용합니다.
    SELECTED_ROOT = CONTROL_ROOT / "followup"
    SELECTION_FILE = SELECTED_ROOT / "selection.json"
elif CAMPAIGN == "paper_baseline":
    SELECTED_ROOT = MODEL_ROOT / f"facex/selected/ucface_paper_baseline_{SURVFACE_PAPER_RECIPE}_v1"
    SELECTION_FILE = SELECTED_ROOT / "plans/selection_v2_paper_magface_only.json"
else:
    # selected는 기존 연구 조건이며 UCFace 동일 재현을 뜻하지 않습니다.
    SELECTED_ROOT = MODEL_ROOT / "facex/selected/ucface_recipes_v1"
    SELECTION_FILE = SELECTED_ROOT / "plans/selection_v2_local_protocol.json"  # 03과 04에서 같은 파일. 목록 변경 시 v2 등 새 파일명.

# 검증 완료 후 원본/전처리 이미지와 목록을 임의로 변경하지 않는다는 전제입니다.
DATA_INTEGRITY_MODE = "immutable_verified_reuse"  # 최초 전체 검증 후 JSON 기록 재사용; "content_verified_reuse"=매번 hash, "full"=매번 hash+decode.
FORCE_DATA_REVERIFY = False  # True=이번 실행에서 전체 이미지 hash+decode 재검증. 성공 후 False로 되돌리세요.
PARENT_SEED = saved["campaign"]["parent_seed"]  # 공통 부모 seed(0 <= seed < 2**32 정수); 후속 head/seed가 공유하므로 02~04에서 유지합니다.
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]  # 부모 학습 epoch 수(양의 정수, 기본 40); 재사용 부모의 완료 epoch와 일치해야 합니다.
PRETRAIN_LR = saved["campaign"]["pretrain_lr"]  # parent의 기존 LR을 별도 유지합니다. downstream 논문 LR과 구분합니다.
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]  # None=02의 고정 경로; 다른 완료 부모를 쓰려면 정확한 경로와 PARENT_SHA256를 함께 지정합니다.
PARENT_SHA256 = saved["campaign"]["parent_sha256"]  # None 또는 지정한 부모 파일의 64자리 SHA-256; 경로와 함께 지정하며 학습 조건도 검증합니다.
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])  # "survface", "tinyface", "scface" 중 선택합니다. LFW/RFW는 평가 전용입니다.
FINETUNE_PROFILES = saved["finetune_profiles"]  # 외부 protocol의 분할·경로·hash입니다. 분할 인물/수량/hash를 여기서 임의로 고치지 않습니다.
AUDIT_WORKERS = 4  # 로컬 release 검사 worker 수(1 이상 정수); 현재 portable 공유 protocol에서는 사용하지 않습니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]  # 03의 보존 설정을 이어받습니다. 04의 export는 학습 checkpoint를 정리하지 않습니다.
NUM_WORKERS = saved["training"]["num_workers"]  # DataLoader worker 수(0~12 정수); 0=현재 프로세스에서 로딩, 큰 값은 CPU/RAM을 더 사용합니다.
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]  # worker당 미리 읽을 batch 수(1 이상 정수); NUM_WORKERS>0일 때 적용합니다.
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]  # True=epoch 사이 worker 유지, False=다시 생성; NUM_WORKERS=0이면 적용되지 않습니다.
TORCH_THREADS = saved["training"]["torch_threads"]  # PyTorch CPU 연산 thread 수(1 이상 정수); GPU 수나 DataLoader worker 수와 다릅니다.
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)  # loss 집계값을 CPU로 가져오는 batch 간격(1 이상 정수); 매 batch 학습 업데이트는 유지합니다.
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]  # 재개용 checkpoint 저장 간격(분, 0보다 큰 값); 짧을수록 복구 간격과 디스크 쓰기가 촘촘해집니다.
DEVICE = saved["project"]["device"]  # 저장된 학습 장치 설정입니다. 이 값을 읽는다고 04에서 GPU 학습이 시작되지는 않습니다.
DATASET = dict(saved["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
BACKBONE = dict(saved["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# 정식 fine-tune: optimizer=adam, lr=MobileFaceNet 0.001 / ResNet-50·100 0.0001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# baseline batch_size=64는 MobileFaceNet·ResNet-50/100 공통; save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(saved["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
HEAD_PARAMETERS = saved["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(saved["parent_head"])  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(saved["augmentation"])  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
ALIGNMENT = dict(saved["alignment"])  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
PATHS = dict(saved["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
EXECUTE = True  # 03의 MagFace 40 epoch 학습 완료 후 True로 실행합니다. 완료 모델만 검증·export하며 새 학습은 수행하지 않습니다.
STAGE = "export"  # 이 노트북의 고정 단계입니다. 다른 단계는 해당 번호의 노트북을 사용합니다.
# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "data_integrity":dict(mode=DATA_INTEGRITY_MODE,force_reverify=FORCE_DATA_REVERIFY),
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,pretrain_lr=PRETRAIN_LR,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS,"parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT})
if CAMPAIGN == "finetune_wd_zero_control":
    from modules.parent_controls import FINETUNE_WD_ZERO_SCHEMA, build_finetune_wd_zero_selection
    from modules.selected_campaign import load_selection, inspect_selection
    if EXECUTE and not SELECTION_FILE.is_file():
        raise FileNotFoundError("03에서 finetune_wd_zero_control 선택 파일을 저장하고 40 epoch 학습을 완료한 뒤 실행하세요.")
    selection = load_selection(SELECTION_FILE) if SELECTION_FILE.is_file() else build_finetune_wd_zero_selection(settings, CONTROL_ROOT, SELECTED_ROOT)
    ft_ctrl = selection["settings"]["campaign"].get("finetune_control", {})
    if (selection["purpose"] != "diagnostic_control"
            or not selection["settings"]["campaign"].get("parent_binding")
            or ft_ctrl.get("schema") != FINETUNE_WD_ZERO_SCHEMA
            or selection["settings"]["training"]["weight_decay"] != 0.0):
        raise ValueError("WD=0 Parent 기반 fine-tune weight_decay=0.0 대조 선택 파일이 필요합니다.")
    settings = selection["settings"]
    TRAINING = dict(settings["training"])
    PARENT_CHECKPOINT = settings["campaign"]["parent_checkpoint"]
    PARENT_SHA256 = settings["campaign"]["parent_sha256"]
elif CAMPAIGN in ("selected", "ucface_review", "paper_baseline", "parent_control"):
    from modules.selected_campaign import load_selection, inspect_selection, make_selection, default_references, ALIGNMENT_CONTROL
    if CAMPAIGN == "parent_control" and not SELECTION_FILE.is_file():
        raise FileNotFoundError("02a의 40 epoch Parent 완료 후 selection.json을 생성하세요.")
    if SELECTION_FILE.exists():
        selection = load_selection(SELECTION_FILE)  # 03에서 확정한 목록을 그대로 읽습니다.
        if CAMPAIGN == "parent_control":
            if selection["purpose"] != "diagnostic_control" or not selection["settings"]["campaign"].get("parent_binding"):
                raise ValueError("02a의 새 Parent 선택 파일이 필요합니다.")
            settings = selection["settings"]
    else:
        plan_rows = paper_baseline_plan(SURVFACE_PAPER_RECIPE, SURVFACE_HEADS, SEEDS) if CAMPAIGN == "paper_baseline" else default_plan()
        purpose = "paper_baseline" if CAMPAIGN == "paper_baseline" else "local_protocol_baseline"
        selection = make_selection(settings, plan_rows, SELECTED_ROOT,
            MODEL_ROOT / "facex/diagnostics" / ALIGNMENT_CONTROL, reuse_completed=REUSE_COMPLETED, purpose=purpose)
        selection["references"] = default_references(selection)
elif CAMPAIGN != "baseline":
    raise ValueError(f"Unknown campaign: {CAMPAIGN}")

from modules.ucface_reference import reproduction_review, require_reproduction_ready


## 2. 대상

In [2]:
# cell 2 : export 대상 계획 및 UCFace 재현 조건 확인
# 목적: export할 데이터셋·head·seed와 필요한 완료 epoch를 확인합니다.

SELECTED_CAMPAIGNS = ("selected", "ucface_review", "paper_baseline", "parent_control", "finetune_wd_zero_control")
plan = inspect_selection(selection, force_reverify=FORCE_DATA_REVERIFY) if CAMPAIGN in SELECTED_CAMPAIGNS else ([{"dataset": d, "stage": "export"} for d in ("scface", "tinyface")]
        if CAMPAIGN == "aligned_controls" else plan_matrix(settings, STAGE))
for row in plan:
    print(row)
print("TARGET_EPOCH:", TRAINING["epoches"], "예정:", len(plan))
if CAMPAIGN == "finetune_wd_zero_control":
    print("단일 변수 대조:", selection["settings"]["campaign"]["finetune_control"])
if CAMPAIGN in SELECTED_CAMPAIGNS:
    print("UCFace 재현 검토:", reproduction_review(selection))


{'run_id': 'survface_magface_full_fr_resize_only_s8972', 'dataset': 'survface', 'head': 'magface', 'recipe': 'full_fr_resize_only', 'seed': 8972, 'status': 'reuse_completed', 'checkpoint': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt', 'target': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt'}
TARGET_EPOCH: 40 예정: 1
단일 변수 대조: {'schema': 'facex-finetune-weight-decay-control-v1', 'purpose': 'diagnostic_control', 'changed_parameter': 'training.weight_decay', 'reference_value': 0.0001, 'control_value': 0.0, 'parent_sha256': '640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213', 'reference_selection': 'F:\\models\\facex\\diagnos

## 3. 출처 기준

In [3]:
# cell 3 : 부모 체크포인트 및 분할 상태 확인
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 공통 부모 정보를 확인하고, 실행 시 고정 분할을 검증해 export용 설정을 저장합니다.

print("공통 부모:", PARENT_CHECKPOINT or "02의 PARENT_SEED 고정 경로", "parent seed:", PARENT_SEED)
print("시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.")
from modules.finetune_protocol import prepare_selected_protocols
if EXECUTE and CAMPAIGN == "baseline":
    for protocol in prepare_selected_protocols(settings, audit_workers=AUDIT_WORKERS):
        print(protocol)
    save_settings(SETTINGS_PATH, settings)
else:
    print("계획 모드: 공식 split 및 전체 이미지 검증/목록 생성 미실행")


공통 부모: F:\models\facex\diagnostics\parent_weight_decay_zero_v1\checkpoint\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt parent seed: 8972
시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.
계획 모드: 공식 split 및 전체 이미지 검증/목록 생성 미실행


## 4. 별도 프로세스로 export 실행

완료 epoch·설정·분할·부모 hash를 검증한 뒤 backbone과 metadata를 게시합니다. 학습은 수행하지 않으며 중단 시 결과 상태를 확인한 뒤 같은 조건으로 다시 실행합니다.


In [4]:
# cell 4 : backbone 가중치 export 및 MODEL_INVENTORY 생성
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 실행 시 완료 checkpoint를 검증해 backbone 가중치와 대응 metadata를 내보냅니다.

from modules.process_runner import launch_campaign
results = []
if EXECUTE and CAMPAIGN in ("selected", "paper_baseline", "parent_control", "finetune_wd_zero_control"):
    results = launch_campaign(BASE_DIR, settings, STAGE, selection=selection, force_reverify=FORCE_DATA_REVERIFY)
elif EXECUTE and CAMPAIGN == "aligned_controls":
    results = [export_alignment_control(settings, row["dataset"], CONTROL_SOURCE, CONTROL_EXPORT) for row in plan]
elif EXECUTE:
    results = launch_campaign(BASE_DIR, settings, STAGE)
else:
    print("계획 또는 benchmark 모드: 정식 작업 미실행")


Child PID 49256; log: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\20c52ac9110949d681842b1258052b8c\console.log; stop request: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\20c52ac9110949d681842b1258052b8c\stop.request
2026-11-21 10:21:53,923 INFO Verified parent checkpoint: F:\models\facex\diagnostics\parent_weight_decay_zero_v1\checkpoint\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt (SHA-256: 640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213)
2026-11-21 10:21:53,924 INFO Parent checkpoint SHA-256 integrity check PASSED.
2026-11-21 10:21:54,270 INFO Verified parent checkpoint: F:\models\facex\checkpoints\shared_v1_boundary_mobilefacenet\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt (SHA-256: 3ac633038bed2bf153e2fe58aae900637881aba8c78c979d22712cc995cf879a)
2026-11-21 10:21:54,270 INFO Parent checkpoint SHA-256 integrity check PASSED.
2026-11-21 10:21:55,824 INFO Verified parent checkpoint: F:\

## 5. 결과

In [5]:
# cell 5 : export 완료 상태 확인
# 목적: export 완료 여부와 결과 경로를 확인하고 ronbun 평가는 별도 단계임을 표시합니다.

for result in results:
    print(result)
if CAMPAIGN in ("selected", "ucface_review", "paper_baseline", "parent_control", "finetune_wd_zero_control") and not EXECUTE:
    print("계획 확인:", {status: sum(r["status"] == status for r in plan) for status in ("reuse_completed", "train_new", "resume", "blocked")})
    print("이번 실행에서 학습/export하지 않았습니다.")
else:
    print("완료 export:", sum(r.get("completed") is True for r in results), "/", len(plan))
if EXECUTE:
    print("strict load/shape 확인. 실제 ronbun·Continuous FIQA 평가는 미실행.")
print("해석 안내:", BASE_DIR / "fine-tune/README.md")


{'run_id': 'survface_magface_full_fr_resize_only_s8972', 'dataset': 'survface', 'head': 'magface', 'recipe': 'full_fr_resize_only', 'seed': 8972, 'completed': True, 'export': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\exports\\survface_magface_full_fr_resize_only_s8972\\survface_magface_full_fr_resize_only_s8972.pt', 'sha256': 'e7def72de05df7ed37de90a13aa460430c4f40dd390d688cd64960f3e3ebdefa', 'source_checkpoint': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt', 'source_sha256': '1dba2a7709570a646de46747aa779489da53e45f0b1cad313685db71c9f46936', 'evaluation_datasets': ['survface'], 'training_dataset': 'survface'}
완료 export: 1 / 1
strict load/shape 확인. 실제 ronbun·Continuous FIQA 평가는 미실행.
해석 안내: f:\FaceX-Zoo\fine-tune\README.md
